In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

train_raw = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test_raw  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout    = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train_raw.merge(layout_clean, on='layout_id', how='left')
test  = test_raw.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'

# 1. 타겟값이 0인 비율과 분포 재확인
print(f"Target 0인 비율: {(train[TARGET]==0).mean():.3f}")
print(f"Target 10분 이하: {(train[TARGET]<=10).mean():.3f}")
print(f"Target 30분 이상: {(train[TARGET]>=30).mean():.3f}")
print(f"Target 100분 이상: {(train[TARGET]>=100).mean():.3f}")

Target 0인 비율: 0.027
Target 10분 이하: 0.530
Target 30분 이상: 0.208
Target 100분 이상: 0.015


In [5]:
# 2. layout_type별 delay 분포
train_raw = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
train_with_layout = train_raw.merge(layout, on='layout_id', how='left')

print(train_with_layout.groupby('layout_type')[TARGET].agg(['mean', 'median', 'std', 'count']))

                  mean     median        std  count
layout_type                                        
grid         18.104791   8.924054  26.248145  91250
hub_spoke    22.280032  11.317745  30.443227  43375
hybrid       18.411890   8.250280  28.282444  73125
narrow       18.360849   9.467351  24.269034  42250


In [6]:
# hub_spoke가 LabelEncoding에 포함됐는지 확인
print(le.classes_)

['grid' 'hub_spoke' 'hybrid' 'narrow']


In [7]:
# 3. layout_type별 test 분포 확인
test_with_layout = test_raw.merge(layout, on='layout_id', how='left')
print(test_with_layout['layout_type'].value_counts())

layout_type
hybrid       20000
grid         16000
narrow        8000
hub_spoke     6000
Name: count, dtype: int64


In [8]:
# 4. train vs test layout_type 분포 비교
train_with_layout = train_raw.merge(layout, on='layout_id', how='left')

train_dist = train_with_layout['layout_type'].value_counts(normalize=True)
test_dist  = test_with_layout['layout_type'].value_counts(normalize=True)

print("Train 비율:")
print(train_dist)
print("\nTest 비율:")
print(test_dist)

Train 비율:
layout_type
grid         0.3650
hybrid       0.2925
hub_spoke    0.1735
narrow       0.1690
Name: proportion, dtype: float64

Test 비율:
layout_type
hybrid       0.40
grid         0.32
narrow       0.16
hub_spoke    0.12
Name: proportion, dtype: float64


In [9]:
# 5. 주요 feature들의 train vs test 분포 비교
features_to_check = ['robot_active', 'order_inflow_15m', 
                     'battery_mean', 'congestion_score',
                     'low_battery_ratio', 'pack_utilization']

for feat in features_to_check:
    print(f"\n[{feat}]")
    print(f"  Train - mean: {train_raw[feat].mean():.3f}, std: {train_raw[feat].std():.3f}")
    print(f"  Test  - mean: {test_raw[feat].mean():.3f}, std: {test_raw[feat].std():.3f}")


[robot_active]
  Train - mean: 13.379, std: 11.449
  Test  - mean: 17.122, std: 14.522

[order_inflow_15m]
  Train - mean: 94.586, std: 77.084
  Test  - mean: 131.676, std: 90.316

[battery_mean]
  Train - mean: 53.455, std: 15.379
  Test  - mean: 50.673, std: 17.422

[congestion_score]
  Train - mean: 9.986, std: 19.516
  Test  - mean: 12.697, std: 21.983

[low_battery_ratio]
  Train - mean: 0.136, std: 0.256
  Test  - mean: 0.187, std: 0.296

[pack_utilization]
  Train - mean: 0.421, std: 0.364
  Test  - mean: 0.461, std: 0.381


In [10]:
# 6. train과 test 분포 차이가 큰 feature 확인
all_features = [c for c in train_raw.columns 
                if c not in ['ID', 'layout_id', 'scenario_id', TARGET]]

shift_report = []
for feat in all_features:
    if train_raw[feat].dtype in [np.float64, np.int64]:
        train_mean = train_raw[feat].mean()
        test_mean  = test_raw[feat].mean()
        if train_mean != 0:
            diff_pct = abs(test_mean - train_mean) / abs(train_mean) * 100
            shift_report.append({'feature': feat, 'train_mean': train_mean, 
                                 'test_mean': test_mean, 'diff_pct': diff_pct})

shift_df = pd.DataFrame(shift_report).sort_values('diff_pct', ascending=False)
print(shift_df.head(20).to_string())

                  feature   train_mean    test_mean   diff_pct
16    charge_queue_length     2.632955     4.025957  52.906389
12      task_reassign_15m     0.152282     0.232438  52.636688
9          robot_charging     6.128704     9.247220  50.883776
17        avg_charge_wait     2.417568     3.556681  47.118129
20       blocked_path_15m     0.463049     0.663214  43.227660
0        order_inflow_15m    94.585662   131.676343  39.213852
22        fault_count_15m     0.336619     0.467046  38.746086
15      low_battery_ratio     0.136184     0.186707  37.098630
21     near_collision_15m     0.383289     0.521553  36.073047
23      avg_recovery_time     0.784342     1.027142  30.955905
1          unique_sku_15m   108.275673   140.775678  30.015980
7            robot_active    13.379388    17.122220  27.974613
18       congestion_score     9.985956    12.696899  27.147561
19       max_zone_density     0.064497     0.079112  22.660509
3      urgent_order_ratio     0.112411     0.130670  16

In [11]:
# test 분포에 가까운 train 샘플 가중치 계산
from scipy.stats import gaussian_kde

# 가장 shift 큰 feature들로 가중치 계산
shift_features = ['charge_queue_length', 'robot_charging', 
                  'order_inflow_15m', 'low_battery_ratio',
                  'avg_charge_wait', 'blocked_path_15m']

# train/test 각 feature 정규화 후 거리 계산
train_shift = train_raw[shift_features].fillna(train_raw[shift_features].median())
test_shift  = test_raw[shift_features].fillna(test_raw[shift_features].median())

# test 평균과 train 각 샘플의 거리 기반 가중치
test_means = test_shift.mean()
distances  = ((train_shift - test_means) ** 2).sum(axis=1)
weights    = np.exp(-distances / distances.std())
weights    = weights / weights.mean()  # 정규화

print(f'가중치 범위: {weights.min():.3f} ~ {weights.max():.3f}')
print(f'가중치 평균: {weights.mean():.3f}')

가중치 범위: 0.000 ~ 2.011
가중치 평균: 1.000


In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import joblib, os
os.makedirs('C:/Users/user/dacon/Smart_Logistics/models', exist_ok=True)

train_raw = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test_raw  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout    = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train_raw.merge(layout_clean, on='layout_id', how='left')
test  = test_raw.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

# Distribution shift 가중치 계산
shift_features = ['charge_queue_length', 'robot_charging',
                  'order_inflow_15m', 'low_battery_ratio',
                  'avg_charge_wait', 'blocked_path_15m']

train_shift = train_raw[shift_features].fillna(train_raw[shift_features].median())
test_shift  = test_raw[shift_features].fillna(test_raw[shift_features].median())

test_means = test_shift.mean()
distances  = ((train_shift - test_means) ** 2).sum(axis=1)
weights    = np.exp(-distances / distances.std())
weights    = weights / weights.mean()

X = train[feature_cols]
y = train[TARGET]

print(f'feature 수: {len(feature_cols)}')
print(f'가중치 범위: {weights.min():.3f} ~ {weights.max():.3f}')

feature 수: 103
가중치 범위: 0.000 ~ 2.011


In [14]:
X = train[feature_cols]
y = train[TARGET]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 가중치 분할
weights_tr = weights.values[X_tr.index]

# 기존 best CatBoost 조건 그대로 (v15와 동일)
cat_params = {
    'iterations'          : 199999,
    'learning_rate'       : 0.03,
    'depth'               : 8,
    'loss_function'       : 'MAE',
    'eval_metric'         : 'MAE',
    'task_type'           : 'GPU',
    'random_seed'         : 42,
    'early_stopping_rounds': 300,
    'verbose'             : 10000,
}

# 가중치 없는 버전
print('=== 가중치 없는 버전 ===')
model_no_w = CatBoostRegressor(**cat_params)
model_no_w.fit(X_tr, y_tr, eval_set=(X_val, y_val))
mae_no_w = np.mean(np.abs(y_val - model_no_w.predict(X_val)))
print(f'Validation MAE: {mae_no_w:.4f}')

# 가중치 있는 버전
print('\n=== 가중치 있는 버전 ===')
model_w = CatBoostRegressor(**cat_params)
model_w.fit(X_tr, y_tr, eval_set=(X_val, y_val), sample_weight=weights_tr)
mae_w = np.mean(np.abs(y_val - model_w.predict(X_val)))
print(f'Validation MAE: {mae_w:.4f}')

print(f'\n차이: {mae_no_w - mae_w:.4f} (양수면 가중치가 효과적)')

=== 가중치 없는 버전 ===


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.5894100	test: 14.6450575	best: 14.6450575 (0)	total: 11.6ms	remaining: 38m 45s
10000:	learn: 8.6015863	test: 8.7394188	best: 8.7394188 (10000)	total: 40.6s	remaining: 12m 51s
20000:	learn: 7.9731500	test: 8.2896981	best: 8.2896981 (20000)	total: 1m 22s	remaining: 12m 26s
30000:	learn: 7.4917906	test: 7.9796962	best: 7.9796962 (30000)	total: 2m 8s	remaining: 12m 6s
40000:	learn: 7.0931675	test: 7.7433987	best: 7.7433987 (40000)	total: 2m 50s	remaining: 11m 23s
50000:	learn: 6.7464638	test: 7.5552544	best: 7.5552544 (50000)	total: 3m 33s	remaining: 10m 40s
60000:	learn: 6.4401275	test: 7.4023694	best: 7.4023694 (60000)	total: 4m 18s	remaining: 10m 4s


KeyboardInterrupt: 